# 🎙️ 1 · Record & prepare your voice

Builds a clean **training dataset** of your voice and saves it to Google Drive (`MyVoice/dataset`).
No GPU needed for this notebook.

**What to record (10-30 minutes total is the sweet spot)**

- Same microphone, same quiet room for every take. No music, no echo, no effects, no auto-tune.
- Mix of **talking and singing**. If you want to sing over your tracks, the singing matters most:
  cover your range (low to high), loud and soft, long notes and fast lines.
- Keep 10-20 cm from the mic and use a pop filter or shirt-sleeve of distance to avoid plosives.
- Leave pauses between sentences; mistakes are fine, but cut out coughs and loud breaths.
- Everything should be **only your voice** (no backing vocals, no other people).

**Folders created on your Drive (`MyVoice/`)**: `recordings` (raw takes) → `dataset` (cleaned) → `models` (trained voice) ·
`tracks` / `instrumentals` / `vocals` (inputs for notebook 3) · `output` (finished songs).

> Only train on voices you have the right to use. This pipeline is meant for **your own** voice.

In [ ]:
#@title 📁 Connect Google Drive and create folders
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

VOICE_ROOT = Path("/content/drive/MyDrive/MyVoice")   # everything lives here, so it survives Colab resets
FOLDERS = {name: VOICE_ROOT / name for name in
           ["recordings", "dataset", "models", "checkpoints", "tracks", "instrumentals", "vocals", "output"]}
for p in FOLDERS.values():
    p.mkdir(parents=True, exist_ok=True)
print("Your folders:")
for name, p in FOLDERS.items():
    print(f"  {name:14s} {p}")

In [ ]:
#@title 🧰 Audio helpers (just run this)
import subprocess
import numpy as np
import soundfile as sf
from pathlib import Path

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".webm", ".mp4", ".mov", ".wma", ".aiff", ".aif"}
SR = 44100
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)

def list_audio(folder):
    return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXTS)

def decode_to_float_wav(src, dst, sr=SR):
    """Any audio/video file -> mono float32 WAV at `sr` (needs ffmpeg, which Colab has)."""
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-vn", "-ac", "1",
                    "-ar", str(sr), "-c:a", "pcm_f32le", str(dst)], check=True)

def frame_db(x, sr, frame_s=0.05):
    n = int(sr * frame_s)
    m = len(x) // n
    if m == 0:
        return np.array([-120.0])
    frames = x[: m * n].reshape(m, n)
    return 20 * np.log10(np.sqrt((frames ** 2).mean(axis=1)) + 1e-9)

def analyse(x, sr):
    """Quick quality numbers for one recording."""
    db = frame_db(x, sr)
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    return {
        "minutes": len(x) / sr / 60,
        "peak_db": 20 * np.log10(max(peak, 1e-9)),
        "clip_pct": 100 * float(np.mean(np.abs(x) >= 0.999)) if len(x) else 0.0,
        "noise_db": float(np.percentile(db, 10)),    # level of the quietest 10% of the file ~ room noise
        "voice_db": float(np.percentile(db, 90)),    # level of the loudest 10% ~ your voice
    }

def trim_silence(x, sr, pad_s=0.25):
    """Cut dead air from the start and end of a take."""
    db = frame_db(x, sr, 0.05)
    thresh = np.percentile(db, 90) - 35
    loud = np.where(db > thresh)[0]
    if len(loud) == 0:
        return x
    start = max(0, int(loud[0] * 0.05 * sr) - int(pad_s * sr))
    end = min(len(x), int((loud[-1] + 1) * 0.05 * sr) + int(pad_s * sr))
    return x[start:end]

def prepare_file(src, dst, denoise=False, denoise_strength=0.6, trim=True, peak_db=-3.0):
    """Clean one recording and save it as a 44.1 kHz mono 16-bit WAV. Returns quality numbers of the ORIGINAL."""
    tmp = WORK / "decoded.wav"
    decode_to_float_wav(src, tmp)
    x, sr = sf.read(tmp, dtype="float32")
    before = analyse(x, sr)
    x = x - np.mean(x)                                   # remove DC offset
    if denoise:
        import noisereduce as nr
        x = nr.reduce_noise(y=x, sr=sr, stationary=True, prop_decrease=denoise_strength).astype(np.float32)
    if trim:
        x = trim_silence(x, sr)
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    if peak > 0:
        x = x * (10 ** (peak_db / 20) / peak)            # peak-normalise
    sf.write(dst, x, sr, subtype="PCM_16")
    before["minutes_after"] = len(x) / sr / 60
    return before

def judge(q):
    """Human-readable warnings for one recording."""
    notes = []
    if q["clip_pct"] > 0.05:
        notes.append("clipping (too loud - re-record with the mic gain lower)")
    if q["noise_db"] > -50:
        notes.append("noisy room (try a quieter spot or turn on denoise)")
    if q["voice_db"] - q["noise_db"] < 25:
        notes.append("voice is not much louder than the background")
    return notes

## Step 1 - get your recordings in
Use any of these (or a mix): record in the browser, upload from your computer, or drop files into the `MyVoice/recordings` Drive folder.

In [ ]:
#@title 🔴 Record with your microphone (browser)
take_name = "take_01"   #@param {type:"string"}
max_minutes = 5         #@param {type:"slider", min:1, max:15, step:1}
# Run this cell once per take. Allow microphone access when your browser asks, talk/sing, then press Stop.
# Tip: for long sessions it is easier to record on your phone / in a DAW and drop the files into
#      MyVoice/recordings on Google Drive (or use the upload cell below).

from base64 import b64decode
from IPython.display import Audio, Javascript, display
from google.colab import output

RECORD_JS = """
const record = (maxMs) => new Promise(async (resolve, reject) => {
  let stream;
  try {
    stream = await navigator.mediaDevices.getUserMedia({audio: {
      echoCancellation: false, noiseSuppression: false, autoGainControl: false, channelCount: 1}});
  } catch (e) { reject(e); return; }
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = (e) => chunks.push(e.data);
  const box = document.createElement('div');
  const btn = document.createElement('button');
  btn.textContent = 'Stop recording';
  btn.style.cssText = 'font-size:18px;padding:10px 18px;cursor:pointer;';
  const label = document.createElement('span');
  label.style.marginLeft = '12px';
  box.appendChild(btn); box.appendChild(label); document.body.appendChild(box);
  const t0 = Date.now();
  const stop = () => { if (rec.state !== 'inactive') rec.stop(); };
  const timer = setInterval(() => {
    label.textContent = 'Recording... ' + Math.floor((Date.now() - t0) / 1000) + 's';
    if (Date.now() - t0 >= maxMs) stop();
  }, 250);
  btn.onclick = stop;
  rec.onstop = () => {
    clearInterval(timer);
    stream.getTracks().forEach((t) => t.stop());
    box.remove();
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(new Blob(chunks, {type: rec.mimeType}));
  };
  rec.start();
});
"""
display(Javascript(RECORD_JS))
data_url = output.eval_js(f"record({int(max_minutes * 60 * 1000)})")

header, payload = data_url.split(",", 1)
ext = ".m4a" if "mp4" in header else ".ogg" if "ogg" in header else ".webm"
out_file = FOLDERS["recordings"] / f"{take_name}{ext}"
out_file.write_bytes(b64decode(payload))
print("Saved", out_file)
display(Audio(str(out_file)))

In [ ]:
#@title ⬆️ (Optional) Upload recordings from your computer
# Any audio format works (wav, mp3, m4a, flac...). You can also just copy files into MyVoice/recordings in Drive.
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    (FOLDERS["recordings"] / name).write_bytes(data)
    print("Saved", FOLDERS["recordings"] / name)

## Step 2 - clean and check them

In [ ]:
#@title 🧼 Clean everything in `recordings` and write the training dataset
denoise = False            #@param {type:"boolean"}
denoise_strength = 0.6     #@param {type:"slider", min:0.2, max:1.0, step:0.1}
trim_silence_ends = True   #@param {type:"boolean"}
clear_dataset_first = True #@param {type:"boolean"}
# Only turn denoise on for steady hiss/fan noise. If the room is quiet, leave it off - denoising can make the cloned voice sound watery.

if clear_dataset_first:
    for old in FOLDERS["dataset"].glob("*.wav"):
        old.unlink()

files_in = list_audio(FOLDERS["recordings"])
if not files_in:
    raise SystemExit(f"No recordings found in {FOLDERS['recordings']}. Record or upload some first.")

total_min = 0.0
problems = 0
print(f"{'file':32s} {'min':>6s} {'peak dB':>8s} {'noise dB':>9s}  notes")
used = set()
for src in files_in:
    stem = src.stem
    while stem in used:                                  # foo.wav + foo.mp3 must not overwrite each other
        stem += "_2"
    used.add(stem)
    dst = FOLDERS["dataset"] / (stem + ".wav")
    q = prepare_file(src, dst, denoise=denoise, denoise_strength=denoise_strength, trim=trim_silence_ends)
    notes = judge(q)
    problems += bool(notes)
    total_min += q["minutes_after"]
    print(f"{src.name[:32]:32s} {q['minutes']:6.1f} {q['peak_db']:8.1f} {q['noise_db']:9.1f}  {'; '.join(notes) or 'ok'}")

print(f"\nDataset: {len(files_in)} file(s), {total_min:.1f} minutes of audio -> {FOLDERS['dataset']}")
if total_min < 10:
    print("⚠️  Under 10 minutes. Cloning works best with 10-30 minutes - record more, ideally including singing.")
elif total_min > 60:
    print("ℹ️  Over an hour. That is more than needed; 20-30 clean minutes usually beats 2 hours of mixed quality.")
else:
    print("✅ Good amount of audio.")
if problems:
    print("⚠️  Some files have issues (see notes above). Re-record or remove them from `recordings`, then run this cell again.")

In [ ]:
#@title 🔊 Listen to what will be used for training
from IPython.display import Audio, display
clips = list_audio(FOLDERS["dataset"])
for clip in clips[:3]:
    print(clip.name)
    display(Audio(str(clip)))
print("Next: open notebook 02_train_voice_model.ipynb")